# Assignment 2

 **This was supoosed to be 2 assignment but is a big assignment So go slow and learn what you are doing have fun**


# Part 1: Data Ingestion

Data Ingestion is the first step in a RAG pipeline. It involves collecting, reading, and loading raw data from various sources (such as PDFs, HTML, or databases) into the system. Here, we read all PDF files in a given directory and parse their content into structured documents containing page content and metadata.


Here we Doing only for pdf but in final project we will do for pdf,csv,xlsx,docx,txt.
if you want you can practise to extract data from one more file format i would love to see you do this.

In [1]:
!pip install langchain langchain-community langchain-text-splitters pypdf pymupdf sentence-transformers chromadb scikit-learn

In [2]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


/tmp/ipykernel_60642/2076300660.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [3]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    documents = []
    # TODO: Implement PDF ingestion using PyPDFLoader.
    # Load all pdf files recursively from pdf_directory (using Path(pdf_directory).glob('**/*.pdf')).
    pdf_files = Path(pdf_directory).glob('**/*.pdf')
    for pdf_file in pdf_files:
        # Load the PDF file using PyPDFLoader
        loader = PyPDFLoader(str(pdf_file))
        loaded_docs = loader.load()
        # Add metadata fields to each loaded document
        for doc in loaded_docs:
            doc.metadata['source_file'] = pdf_file.name
            doc.metadata['file_type'] = 'pdf'
            documents.append(doc)
    # For each page document loaded, add metadata fields:
    # - 'source_file': the filename of the PDF file
    # - 'file_type': 'pdf'
    # Collect all loaded documents in a list and return them.
    # Hint: Use PyPDFLoader(str(pdf_file)).load()
    return documents


In [4]:
all_pdf_documents = process_all_pdfs("/content/DATA")
all_pdf_documents

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '2014-03-15T13:38:54+01:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2014-03-15T13:38:54+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/MacPorts 2013_5) kpathsea version 6.1.1', 'source': '/content/DATA/scan.pdf', 'total_pages': 15, 'page': 0, 'page_label': 'i', 'source_file': 'scan.pdf', 'file_type': 'pdf'}, page_content='A Scandal in Bohemia\nArthur Conan Doyle'),
 Document(metadata={'producer': '', 'creator': '', 'creationdate': '2014-03-15T13:38:54+01:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2014-03-15T13:38:54+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/MacPorts 2013_5) kpathsea version 6.1.1', 'source': '/content/DATA/scan.pdf', 'total_pages': 15, 'page': 1, 'page_label': 'ii', 'source_file': 'scan.pdf', 'file_type': 'pdf'},

# Part 2: Chunking

Chunking is the process of breaking down large documents into smaller, cohesive segments (chunks). Since Large Language Models (LLMs) have a limited context window (input size limit) and retrieve information more accurately from smaller context blocks, we must split large documents. In this assignment, you need to use **RecursiveCharacterTextSplitter** to split loaded documents into smaller paragraphs with proper overlap to maintain context between boundary lines.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )

    # splitting
    chunks = text_splitter.split_documents(documents)

    print(f"Successfully split {len(documents)} document pages into {len(chunks)} chunks.")
    return chunks


In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Successfully split 15 document pages into 63 chunks.


[Document(metadata={'producer': '', 'creator': '', 'creationdate': '2014-03-15T13:38:54+01:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2014-03-15T13:38:54+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/MacPorts 2013_5) kpathsea version 6.1.1', 'source': '/content/DATA/scan.pdf', 'total_pages': 15, 'page': 0, 'page_label': 'i', 'source_file': 'scan.pdf', 'file_type': 'pdf'}, page_content='A Scandal in Bohemia\nArthur Conan Doyle'),
 Document(metadata={'producer': '', 'creator': '', 'creationdate': '2014-03-15T13:38:54+01:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2014-03-15T13:38:54+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/MacPorts 2013_5) kpathsea version 6.1.1', 'source': '/content/DATA/scan.pdf', 'total_pages': 15, 'page': 1, 'page_label': 'ii', 'source_file': 'scan.pdf', 'file_type': 'pdf'},

# Part 3: Embedding

Embedding is the process of converting text blocks into numerical vector representations. These vectors capture the semantic meaning of the text. Sentences that are semantically similar will be closer to each other in the vector space. We use pre-trained sentence transformer models (like 'all-MiniLM-L6-v2') to convert text chunks and queries into embeddings.

---



from sentence_transformers import SentenceTransformer

Imports the embedding model class.

This library:

downloads pretrained transformer models
converts text → embeddings

Built on top of:

HuggingFace transformers
PyTorch

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid #to get each chunk a unique id after embedding
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        print(f"Loading embedding model: '{self.model_name}'...")
        self.model = SentenceTransformer(self.model_name)
        embedding_dim = self.model.get_sentence_embedding_dimension()
        print(f"Successfully loaded model! Embedding dimension size: {embedding_dim}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if self.model is None:
            raise RuntimeError("SentenceTransformer model is not initialized.")

        # Encode the texts and return the numpy array
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings



# Part 4: Vector DB

Vector Database (Vector DB) is a specialized database designed to store and query high-dimensional vector embeddings efficiently. It allows us to persist our document chunks along with their computed embeddings and perform fast search operations. Here, we use ChromaDB, which runs locally and stores documents persistently in a directory.

In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        os.makedirs(self.persist_directory, exist_ok=True)

        # Initialize client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # Get or create collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "Vector store containing PDF chunks and embeddings"}
        )

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must exactly match the number of embeddings.")

        ids = []
        metadatas = []
        contents = []

        # Convert NumPy array of embeddings to a list of lists for Chroma
        embeddings_list = embeddings.tolist()

        for idx, doc in enumerate(documents):
            # Generate unique ID
            doc_id = uuid.uuid4().hex[:8]
            ids.append(doc_id)

            # Extract content string
            content = doc.page_content
            contents.append(content)

            # Create metadata dict safely
            metadata = doc.metadata.copy() if doc.metadata else {}
            metadata['doc_index'] = idx
            metadata['content_length'] = len(content)
            metadatas.append(metadata)

        # Add to the collection
        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=contents
        )


In [11]:
chunks = split_documents(all_pdf_documents)
chunks

Successfully split 15 document pages into 63 chunks.


[Document(metadata={'producer': '', 'creator': '', 'creationdate': '2014-03-15T13:38:54+01:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2014-03-15T13:38:54+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/MacPorts 2013_5) kpathsea version 6.1.1', 'source': '/content/DATA/scan.pdf', 'total_pages': 15, 'page': 0, 'page_label': 'i', 'source_file': 'scan.pdf', 'file_type': 'pdf'}, page_content='A Scandal in Bohemia\nArthur Conan Doyle'),
 Document(metadata={'producer': '', 'creator': '', 'creationdate': '2014-03-15T13:38:54+01:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2014-03-15T13:38:54+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/MacPorts 2013_5) kpathsea version 6.1.1', 'source': '/content/DATA/scan.pdf', 'total_pages': 15, 'page': 1, 'page_label': 'ii', 'source_file': 'scan.pdf', 'file_type': 'pdf'},

## convert text to embeddings


In [12]:
### lets see text of chumks
texts=[doc.page_content for doc in chunks]

texts

['A Scandal in Bohemia\nArthur Conan Doyle',
 'This text is provided to you “as-is” without any warranty. No warranties of any kind, expressed or implied, are made to you as to the\ntext or any medium it may be on, including but not limited to warranties of merchantablity or ﬁtness for a particular purpose.\nThis text was formatted from various free ASCII and HTML variants. See http:/ /sherlock-holm.es for an electronic form of this text\nand additional information about it.\nThis text comes from the collection’s version 3.1.',
 'T able of contents\nChapter 1 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .3\nChapter 2 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .7\nChapter 3 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .12\n1',
 'CHAPTER I.\nT\no Sherlock Holmes sh

In [13]:
## Generate the Embeddings
embedding_manager = EmbeddingManager()
embeddings=embedding_manager.generate_embeddings(texts)
vectorstore = VectorStore()
##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Loading embedding model: 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully loaded model! Embedding dimension size: 384


/tmp/ipykernel_60642/906996155.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = self.model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

# Part 5: Query Retrieval

Query Retrieval starts with the user entering a natural language query. We must convert this query into the same embedding space using our embedding manager. This encoded query is then sent to the vector database to retrieve raw results.

---

# Part 6: Similarity Search

Similarity Search is the mathematical calculation (such as Cosine Similarity) used by the vector store to compare the query embedding with document embeddings. It ranks and returns the top_k most similar documents. We can filter results by a minimum similarity score (score_threshold) to keep only relevant context.


In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        # 1. Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0].tolist()

        # 2. Query the vector store
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )

        retrieved_docs = []

        # ChromaDB returns a list of lists.
        if not results['ids'] or not results['ids'][0]:
            return retrieved_docs

        # 3. Compile results WITHOUT the strict threshold filter
        for i in range(len(results['ids'][0])):
            distance = results['distances'][0][i]

            # A mathematically safe way to calculate similarity (never goes negative)
            similarity_score = 1 / (1 + distance)

            # We are ignoring the score_threshold right now to guarantee you see output!
            retrieved_docs.append({
                'id': results['ids'][0][i],
                'content': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'similarity_score': similarity_score,
                'distance': distance,
                'rank': i + 1
            })

        return retrieved_docs

In [15]:
rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)

In [16]:
rag_retriever.retrieve("What is Multivac?", top_k=3)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[{'id': '1f7e9828',
  'content': 'entropy1 can be reversed.H e takes the challenge of 5 dollar bet w ith Lupow and poses\nthe question to M ultivac: “H ow can the net am ount of entropy of the universe be\nm assively decreased?” M ultivac responds w ith “IN SU FFIC IEN T D A T A FO R\nM EA N IN G FU L A N SW ER ”.\nScene 2\nM IC R O V A C\n(Jerrodd,Jerrodine,and Jerrodette Iand II)\nT he story skips ahead a long period of tim e and the second scene is set aw ay\nfrom Earth inspace. W e are introduced w ith a new generation fam ily of four m em bers\n1Entropy: the am ountof (energy) running-dow n of the universe.',
  'metadata': {'source_file': 'Asimov (The Last Question) (1).pdf',
   'source': '/content/DATA/Asimov (The Last Question) (1).pdf',
   'content_length': 576,
   'producer': 'PyPDF',
   'total_pages': 3,
   'page': 0,
   'creator': 'PyPDF',
   'creationdate': '',
   'doc_index': 2,
   'file_type': 'pdf',
   'page_label': '1'},
  'similarity_score': 0.37007804957415874,
  'dis

# Part 7: Prompting and Calling LLM

Prompting and Calling LLM is the synthesis step of RAG. We take the retrieved contexts, format them into a structured prompt alongside the user's original query, and pass them to the Large Language Model (LLM) to generate a grounded, factually accurate response.


In [17]:
!pip install -q langchain-groq

In [18]:
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(
    groq_api_key=os.environ["GROQ_API_KEY"],
    model_name="llama-3.1-8b-instant",   # ✅ current working model
    temperature=0.1,
    max_tokens=512
)

# Quick test
test = llm.invoke([HumanMessage(content="Say hello in one sentence.")])
print("✅ LLM ready! Test:", test.content)

✅ LLM ready! Test: Hello, how can I assist you today?


In [19]:
def rag_simple(query, retriever, llm, top_k=3):
    """Simple RAG pipeline — retrieval + LLM generation."""
    # 1. Retrieve top_k relevant chunks
    docs = retriever.retrieve(query, top_k=top_k)

    # 2. Build context string
    context = "\n\n---\n\n".join([doc['content'] for doc in docs])

    # 3. Format prompt
    prompt_text = (
        f"You are a helpful assistant. Use the context below to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )

    # 4. Call LLM
    response = llm.invoke([HumanMessage(content=prompt_text)])
    return response.content


# Run it
answer = rag_simple(
    query="Who is Irene Adler?",
    retriever=rag_retriever,
    llm=llm,
    top_k=3
)
print("--- FINAL ANSWER ---")
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- FINAL ANSWER ---
Irene Adler is a well-known adventuress, a former opera singer, and a prima donna who was born in New Jersey in 1858. She was a contralto who performed at La Scala and the Imperial Opera of Warsaw, and later retired from the operatic stage.


# Part 8: Advanced RAG (Optional)

Advanced RAG includes sophisticated pipeline elements such as streaming responses, citation tracking, interaction history (conversational memory), response summarization, and score-based filtering to make the application robust and production-ready.


In [20]:
from langchain_core.messages import HumanMessage

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    # 1. Retrieve top_k documents using score threshold min_score
    docs = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    # 2. Parse sources including source_file, page, similarity_score, and a content preview
    sources = []
    context_chunks = []

    for doc in docs:
        meta = doc.get('metadata', {})
        sources.append({
            'source_file':      meta.get('source_file', 'Unknown'),
            'page':             meta.get('page', 'Unknown'),
            'similarity_score': doc.get('similarity_score', 0.0),
            'preview':          doc['content'][:100].replace('\n', ' ') + "..."
        })
        context_chunks.append(
            f"[Source: {meta.get('source_file')}, Page: {meta.get('page')}]\n{doc['content']}"
        )

    context = "\n\n".join(context_chunks)

    # 3. Determine confidence (max similarity score of retrieved docs)
    confidence = docs[0]['similarity_score'] if docs else 0.0

    # 4. Invoke LLM with formatted prompt combining context and query
    prompt = (
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        f"Answer comprehensively based on the context:"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    answer   = response.content

    # 5. Return dict containing 'answer', 'sources', 'confidence' (and 'context' if return_context is True)
    result = {
        'answer':     answer,
        'sources':    sources,
        'confidence': confidence
    }

    if return_context:
        result['context'] = context

    return result


# ── Test it ──
adv_result = rag_advanced(
    query="What trick did Holmes use to make Irene Adler reveal the hiding spot?",
    retriever=rag_retriever,
    llm=llm,
    top_k=5,
    return_context=False
)
print("--- ADVANCED ANSWER ---")
print(adv_result['answer'])
print("\n--- SOURCES ---")
for s in adv_result['sources']:
    print(f"  • {s['source_file']} | Page {s['page']} | Score: {s['similarity_score']:.3f}")
    print(f"    Preview: {s['preview']}")
print(f"\nConfidence: {adv_result['confidence']:.3f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- ADVANCED ANSWER ---
Unfortunately, the provided context does not mention Holmes using a trick to make Irene Adler reveal the hiding spot. In fact, the context suggests the opposite: Irene Adler outwitted Holmes, and her "woman's wit" foiled his plans.

However, based on the broader context of the story, it is likely that the story is referring to the case of "A Scandal in Bohemia," where Irene Adler, a opera singer, outwits Holmes by disguising herself as a man and escaping with the King's photograph.

--- SOURCES ---
  • scan.pdf | Page 14 | Score: 0.569
    Preview: away without observing the hand which the King had stretched out to him, he set off in my company fo...
  • scan.pdf | Page 14 | Score: 0.569
    Preview: away without observing the hand which the King had stretched out to him, he set off in my company fo...
  • scan.pdf | Page 14 | Score: 0.569
    Preview: away without observing the hand which the King had stretched out to him, he set off in my company fo...
  • sca

In [21]:
from typing import List, Dict, Any
from langchain_core.messages import HumanMessage
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm       = llm
        self.history   = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2,
              stream: bool = False, summarize: bool = False) -> Dict[str, Any]:

        # 1. Retrieve documents using self.retriever
        docs = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)

        # 2. Parse sources/metadata and construct the context
        sources = []
        context_chunks = []

        for doc in docs:
            meta = doc.get('metadata', {})
            sources.append({
                'source_file': meta.get('source_file', 'Unknown'),
                'page':        meta.get('page', 'Unknown')
            })
            context_chunks.append(
                f"[Source: {meta.get('source_file')}, Page: {meta.get('page')}]\n{doc['content']}"
            )

        context = "\n\n".join(context_chunks)
        prompt  = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"

        # 3. If stream=True, simulate streaming by printing the prompt in chunks
        if stream:
            print("⟳ Streaming prompt preview:\n")
            for word in question.split():
                print(word, end=" ", flush=True)
                time.sleep(0.05)
            print("\n" + "=" * 40)

        # 4. Generate the answer using self.llm
        response   = self.llm.invoke([HumanMessage(content=prompt)])
        raw_answer = response.content

        # 5. Construct citations list and append it to the answer
        citations = "\n\n**Citations:**\n" + "\n".join(
            [f"- {s['source_file']} (Page {s['page']})" for s in sources]
        )
        final_answer = raw_answer + citations

        # 6. If summarize=True, call the LLM to get a 2-sentence summary of the answer
        summary = None
        if summarize:
            sum_response = self.llm.invoke([HumanMessage(
                content=f"Summarize the following answer in exactly 2 sentences:\n\n{raw_answer}"
            )])
            summary = sum_response.content

        # 7. Append the question, answer, sources, and summary to self.history
        record = {
            'question': question,
            'answer':   final_answer,
            'sources':  sources,
            'summary':  summary
        }
        self.history.append(record)

        # 8. Return dictionary containing question, answer (with citations), sources, summary, and history
        return {
            'question': question,
            'answer':   final_answer,
            'sources':  sources,
            'summary':  summary,
            'history':  self.history[:-1]  # return prior history (not current)
        }


# ── Test the full pipeline ──
pipeline = AdvancedRAGPipeline(retriever=rag_retriever, llm=llm)

result = pipeline.query(
    question="What was written in the letter Irene left for Sherlock Holmes?",
    top_k=5,
    summarize=True,
    stream=True
)

print("--- ANSWER ---")
print(result['answer'])
print("\n--- 2-SENTENCE SUMMARY ---")
print(result['summary'])
print("\n--- SOURCES ---")
for s in result['sources']:
    print(f"  • {s['source_file']} | Page {s['page']}")

# ── Test conversational history ──
result2 = pipeline.query(
    question="Why did Irene Adler decide to flee England?",
    top_k=3,
    summarize=False
)
print("\n--- SECOND QUERY ANSWER ---")
print(result2['answer'])
print(f"\nHistory so far: {len(pipeline.history)} queries stored")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

⟳ Streaming prompt preview:

What was written in the letter Irene left for Sherlock Holmes? 
--- ANSWER ---
Unfortunately, the provided text does not mention the content of the letter Irene Adler left for Sherlock Holmes. However, it does mention that Irene Adler left a letter for Sherlock Holmes, which is a crucial part of the story.

**Citations:**
- scan.pdf (Page 14)
- scan.pdf (Page 14)
- scan.pdf (Page 14)
- scan.pdf (Page 7)
- scan.pdf (Page 7)

--- 2-SENTENCE SUMMARY ---
The provided text does not mention the content of the letter Irene Adler left for Sherlock Holmes. However, it does confirm that Irene Adler left a letter for Sherlock Holmes, which is an important part of the story.

--- SOURCES ---
  • scan.pdf | Page 14
  • scan.pdf | Page 14
  • scan.pdf | Page 14
  • scan.pdf | Page 7
  • scan.pdf | Page 7


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- SECOND QUERY ANSWER ---
The provided text does not explicitly state why Irene Adler decided to flee England. However, based on the context, it can be inferred that Irene Adler is the same character as the "lady" mentioned in the text, and she is being referred to as Irene Adler in the narrative. 

In the original Sherlock Holmes story "A Scandal in Bohemia" by Sir Arthur Conan Doyle, Irene Adler is a character who outwits Sherlock Holmes and escapes to England. However, the text snippet provided does not mention her decision to flee England. 

To answer the question, we would need more context or information from the original story.

**Citations:**
- scan.pdf (Page 11)
- scan.pdf (Page 11)
- scan.pdf (Page 11)

History so far: 2 queries stored
